# 02 - 飞书文档 CRUD 测试

测试内容:
1. 创建文档 (POST /docx/v1/documents)
2. 读取文档纯文本 (GET /docx/v1/documents/{id}/raw_content)
3. 获取文档块结构 (GET /docx/v1/documents/{id}/blocks/{id}/children)
4. 追加内容块 (POST /docx/v1/documents/{id}/blocks/{id}/children)
5. 更新块内容 (PATCH /docx/v1/documents/{id}/blocks/{block_id})
6. 删除块 (DELETE /docx/v1/documents/{id}/blocks/{block_id})
7. 授予权限 (POST /drive/v1/permissions/{id}/members)

参考文档:
- 创建文档: https://open.feishu.cn/document/server-docs/docs/docs/docx-v1/document/create
- 读取内容: https://open.feishu.cn/document/server-docs/docs/docs/docx-v1/document/raw_content
- 块操作: https://open.feishu.cn/document/server-docs/docs/docs/docx-v1/blocks/overview

In [1]:
import os
import json
from pathlib import Path

# 加载 .env
try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
except ImportError:
    pass

from feishu_client import FeishuClient, md_to_blocks, make_text_block, make_heading_block, make_code_block, extract_text_from_block, block_type_name

client = FeishuClient()
print("✓ 客户端初始化成功")

✓ 客户端初始化成功


## 2.1 创建文档

In [2]:
# 创建一个空文档
result = client.api('POST', '/docx/v1/documents', json_data={
    'title': 'API 测试文档 - 由 Python 创建'
})

doc = result['document']
DOCUMENT_ID = doc['document_id']

print(f"文档创建成功!")
print(f"  document_id: {DOCUMENT_ID}")
print(f"  title: {doc.get('title')}")
print(f"  url: https://feishu.cn/docx/{DOCUMENT_ID}")
print(f"\n⚠ 请保存这个 DOCUMENT_ID，后续测试会用到")

文档创建成功!
  document_id: CrUYdmbNEoLaUOxaVUmcRE4Ln6g
  title: API 测试文档 - 由 Python 创建
  url: https://feishu.cn/docx/CrUYdmbNEoLaUOxaVUmcRE4Ln6g

⚠ 请保存这个 DOCUMENT_ID，后续测试会用到


## 2.2 追加内容（Markdown → 块）

In [3]:
markdown_content = """
# 欢迎使用飞书文档 API

这是一段普通文本，用于测试文档创建和追加功能。

## 功能列表

- 创建文档
- 读取内容
- 追加块
- 更新块
- 删除块

## 代码示例

print('Hello Feishu!')

> 这是一段引用文本
"""

blocks = md_to_blocks(markdown_content)
print(f"Markdown 转换为 {len(blocks)} 个块")

# 追加到文档
result = client.api(
    'POST',
    f'/docx/v1/documents/{DOCUMENT_ID}/blocks/{DOCUMENT_ID}/children',
    json_data={'children': blocks}
)

print(f"✓ 追加成功! 块数量: {len(blocks)}")

Markdown 转换为 11 个块
✓ 追加成功! 块数量: 11


## 2.3 读取文档纯文本

In [4]:
result = client.api('GET', f'/docx/v1/documents/{DOCUMENT_ID}/raw_content')

print("=== 文档纯文本内容 ===")
print(result.get('content', '无内容'))

=== 文档纯文本内容 ===
API 测试文档 - 由 Python 创建
欢迎使用飞书文档 API
这是一段普通文本，用于测试文档创建和追加功能。
功能列表
创建文档
读取内容
追加块
更新块
删除块
代码示例
print('Hello Feishu!')
这是一段引用文本



## 2.4 获取文档块结构

In [5]:
result = client.api(
    'GET',
    f'/docx/v1/documents/{DOCUMENT_ID}/blocks/{DOCUMENT_ID}/children',
    params={'page_size': 500}
)

items = result.get('items', [])
print(f"文档共有 {len(items)} 个块\n")

for i, item in enumerate(items):
    block_id = item['block_id']
    block_type = item['block_type']
    type_name = block_type_name(block_type)
    text = extract_text_from_block(item)
    print(f"{i+1}. [{type_name:12}] {block_id} | {text[:60]}")

文档共有 11 个块

1. [heading1    ] doxcnNkvsi0nD7RbwUupOl4p9nb | 欢迎使用飞书文档 API
2. [text        ] doxcndIBmg8h4HuVnJoX4LJFoFf | 这是一段普通文本，用于测试文档创建和追加功能。
3. [heading2    ] doxcnNaa4GdFFfXkJq7SPFVnUMf | 功能列表
4. [bullet      ] doxcnfExkyPnQ8L4yYqPPbYlqde | 创建文档
5. [bullet      ] doxcndXbRRQiuv3Vx1AOEVnKhPK | 读取内容
6. [bullet      ] doxcnHU0QpSCJOYW1XE4uaHQl4b | 追加块
7. [bullet      ] doxcnCySYklTHvQts3ltGAOkiMf | 更新块
8. [bullet      ] doxcnNYPCIrpqt2PFJxjvdN63cc | 删除块
9. [heading2    ] doxcnfpumkuQbV10JegDjYtpsjc | 代码示例
10. [text        ] doxcnw6Kqw6eXRnFhKaK8NzPyUg | print('Hello Feishu!')
11. [quote       ] doxcn9sNxDLccMSBC1sS9gmagZc | 这是一段引用文本


## 2.5 更新指定块

In [6]:
# 找到第一个 text 块并更新它
text_block = None
for item in items:
    if item['block_type'] == 2:
        text_block = item
        break

if text_block:
    block_id = text_block['block_id']
    old_text = extract_text_from_block(text_block)
    new_text = old_text + " [已更新 by API]"
    
    # PATCH /blocks/{block_id}，直接传 update_text_elements（无 blocks 包装）
    client.api(
        'PATCH',
        f'/docx/v1/documents/{DOCUMENT_ID}/blocks/{block_id}',
        json_data={
            'update_text_elements': {
                'elements': [
                    {'text_run': {'content': new_text, 'text_element_style': {}}}
                ]
            }
        }
    )
    print(f"✓ 块已更新: {old_text[:40]}... → {new_text[:40]}...")
else:
    print("未找到 text 块")

✓ 块已更新: 这是一段普通文本，用于测试文档创建和追加功能。... → 这是一段普通文本，用于测试文档创建和追加功能。 [已更新 by API]...


## 2.6 删除指定块

In [7]:
# 删除最后一个块（通过父块 /children/batch_delete 按索引删）
if len(items) > 1:
    last_block = items[-1]
    last_text = extract_text_from_block(last_block)
    last_index = len(items) - 1
    
    # DELETE /blocks/{parent_id}/children/batch_delete
    # parent_id 就是文档根 Page 的 block_id，即 DOCUMENT_ID
    client.api(
        'DELETE',
        f'/docx/v1/documents/{DOCUMENT_ID}/blocks/{DOCUMENT_ID}/children/batch_delete',
        json_data={
            'start_index': last_index,
            'end_index': last_index + 1
        }
    )
    print(f"✓ 已删除最后一块: [{block_type_name(last_block['block_type'])}] {last_text[:40]}...")
else:
    print("块数量太少，不删除")

✓ 已删除最后一块: [quote] 这是一段引用文本...


## 2.7 读取更新后的文档

In [8]:
result = client.api('GET', f'/docx/v1/documents/{DOCUMENT_ID}/raw_content')

print("=== 更新后的文档内容 ===")
print(result.get('content', '无内容'))

=== 更新后的文档内容 ===
API 测试文档 - 由 Python 创建
欢迎使用飞书文档 API
这是一段普通文本，用于测试文档创建和追加功能。 [已更新 by API]
功能列表
创建文档
读取内容
追加块
更新块
删除块
代码示例
print('Hello Feishu!')



## 2.8 授予权限（可选）

In [9]:
# 授予某个用户编辑权限
# 需要知道用户的 open_id
OWNER_OPEN_ID = os.environ.get('FEISHU_OWNER_OPEN_ID')

if OWNER_OPEN_ID:
    try:
        client.api(
            'POST',
            f'/drive/v1/permissions/{DOCUMENT_ID}/members',
            json_data={
                'member_type': 'openid',
                'member_id': OWNER_OPEN_ID,
                'perm': 'full_access'
            },
            params={'type': 'docx', 'need_notification': 'false'}
        )
        print(f"✓ 已授予 {OWNER_OPEN_ID} 编辑权限")
    except Exception as e:
        print(f"⚠ 授权失败: {e}")
else:
    print("未设置 FEISHU_OWNER_OPEN_ID，跳过授权")

未设置 FEISHU_OWNER_OPEN_ID，跳过授权
